# Activation Space vs. Behavior Space: A Goodfire Demo

> **Educational miniature** of the Goodfire Manifold Steering approach (arXiv:2605.05115).
> Not a full paper reproduction — an illustration of the core geometric claim on GPT-2.

**Core claim (Goodfire):** The geometry of how a model *internally represents* concepts
(activation space) approximately mirrors the geometry of how it *behaves* on those concepts
(behavior space). This approximate match is called an **isometry**.

**Model:** GPT-2 (local, ~500 MB, no API key)
**Concepts:** Days of the week — an ordered cyclic set with clear sequential structure in
GPT-2's training data. This matches the example used in the Goodfire paper.

In [ ]:
import warnings
from itertools import combinations
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

from prompts import CONCEPT_PROMPTS, DAYS

print(f"Concepts: {DAYS}")
print(f"Total prompts: {sum(len(v) for v in CONCEPT_PROMPTS.values())}")

## Section 1: What Is Approximate Isometry? (Synthetic)

Before loading any model, we understand "approximate isometry" geometrically.

Synthetic activation and behavior centroids are arranged in a cycle (like days of the week),
related by a noisy linear map. The isometry check: do their pairwise distances correlate?

If the scatter of (d_activation, d_behavior) for all concept pairs falls near a line,
the two spaces are approximately isometric — same shape, possibly different scale/rotation.

In [ ]:
np.random.seed(42)
N = 7
short_labels = [d[:3] for d in DAYS]
angles    = np.linspace(0, 2 * np.pi, N, endpoint=False)
act_synth = np.column_stack([np.cos(angles), np.sin(angles)]) * 2.0

A         = np.array([[0.8, -0.5], [0.3, 1.2]])
beh_synth = (A @ act_synth.T).T + np.random.randn(N, 2) * 0.18

colors_synth = [plt.cm.tab10(i / N) for i in range(N)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, data, title in [
    (axes[0], act_synth, "Synthetic Activation Space"),
    (axes[1], beh_synth, "Synthetic Behavior Space"),
]:
    for i in range(N):
        ax.scatter(*data[i], color=colors_synth[i], s=130, zorder=3)
        ax.annotate(short_labels[i], data[i] + np.array([0.08, 0.08]), fontsize=10)
    for i in range(N):
        j = (i + 1) % N
        ax.plot([data[i,0], data[j,0]], [data[i,1], data[j,1]],
                color="gray", alpha=0.4, lw=1.5)
    ax.set_title(title, fontsize=12)
    ax.set_aspect("equal")
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")

plt.suptitle("Same cyclic concept geometry — two different spaces", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
pair_idxs   = list(combinations(range(N), 2))
d_act_synth = np.array([np.linalg.norm(act_synth[i]-act_synth[j]) for i,j in pair_idxs])
d_beh_synth = np.array([np.linalg.norm(beh_synth[i]-beh_synth[j]) for i,j in pair_idxs])
adj_synth   = np.array([abs(i-j)==1 or abs(i-j)==N-1 for i,j in pair_idxs])

r_synth, _ = pearsonr(d_act_synth, d_beh_synth)
m_synth     = np.polyfit(d_act_synth, d_beh_synth, 1)
x_line      = np.linspace(d_act_synth.min(), d_act_synth.max(), 100)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(d_act_synth[adj_synth],  d_beh_synth[adj_synth],
           color="steelblue", s=80, alpha=0.9, label="Adjacent pairs")
ax.scatter(d_act_synth[~adj_synth], d_beh_synth[~adj_synth],
           color="coral",     s=60, alpha=0.7, label="Non-adjacent pairs")
ax.plot(x_line, np.polyval(m_synth, x_line), "k--", lw=1.5,
        label=f"fit (slope={m_synth[0]:.2f})")
ax.set_xlabel("d_activation (synthetic)", fontsize=11)
ax.set_ylabel("d_behavior (synthetic)",   fontsize=11)
ax.set_title(f"Pairwise Distance Scatter\nPearson r = {r_synth:.3f}", fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Synthetic Pearson r = {r_synth:.4f}  (A is non-isometric + noise: approximate isometry by construction)")

## Section 2: Activation Space (GPT-2 Hidden States)

For each prompt we extract GPT-2's final-layer hidden state at the last token position
(768-dim vector) — the model's internal representation just before deciding what to output.

Pipeline: tokenize -> run with output_hidden_states=True -> hidden_states[-1][0, -1, :]
-> 768D vector -> 105 vectors total -> PCA (10D for distances, 2D for visualization) -> centroids.

In [ ]:
print("Loading GPT-2 (downloads ~500MB on first run, then cached)...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model     = GPT2LMHeadModel.from_pretrained("gpt2")
model.train(False)   # set to inference mode
print(f"Model loaded. Hidden dim: {model.config.n_embd}, Vocab: {model.config.vocab_size}")

In [ ]:
def get_hidden_state(prompt: str) -> np.ndarray:
    """Final-layer hidden state at last token position. Shape: (768,)"""
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    # hidden_states is a tuple: (embedding + one per transformer layer)
    # Index -1 is the final transformer block output; shape (1, seq_len, 768)
    return outputs.hidden_states[-1][0, -1, :].numpy()

print("Extracting hidden states (105 prompts)...")
hidden_vectors, hidden_labels = [], []
for day in DAYS:
    for prompt in CONCEPT_PROMPTS[day]:
        hidden_vectors.append(get_hidden_state(prompt))
        hidden_labels.append(day)

hidden_vectors = np.array(hidden_vectors)   # (105, 768)
hidden_labels  = np.array(hidden_labels)
print(f"Collected: {hidden_vectors.shape}")

pca_act_10d = PCA(n_components=10, random_state=42)
pca_act_2d  = PCA(n_components=2,  random_state=42)
act_10d = pca_act_10d.fit_transform(hidden_vectors)
act_2d  = pca_act_2d.fit_transform(hidden_vectors)

act_centroids_10d = {day: act_10d[hidden_labels == day].mean(axis=0) for day in DAYS}
act_centroids_2d  = {day: act_2d[hidden_labels == day].mean(axis=0)  for day in DAYS}

print(f"PCA done. Explained variance (10 PCs): {pca_act_10d.explained_variance_ratio_.sum():.1%}")

In [ ]:
colors_map = {day: plt.cm.tab10(i / 7) for i, day in enumerate(DAYS)}
short      = {d: d[:3] for d in DAYS}

fig, ax = plt.subplots(figsize=(7, 6))
for day in DAYS:
    mask = hidden_labels == day
    ax.scatter(act_2d[mask, 0], act_2d[mask, 1],
               color=colors_map[day], alpha=0.25, s=30)
for day in DAYS:
    ax.scatter(*act_centroids_2d[day], color=colors_map[day], s=180, zorder=4,
               edgecolors="white", linewidths=1.5, label=short[day])
for i in range(len(DAYS)):
    j = (i + 1) % len(DAYS)
    c1, c2 = act_centroids_2d[DAYS[i]], act_centroids_2d[DAYS[j]]
    ax.plot([c1[0], c2[0]], [c1[1], c2[1]], color="gray", lw=1.2, alpha=0.5)

ax.set_title("Activation Space — GPT-2 final hidden states (PCA 2D)\nCentroids connected in day-of-week order", fontsize=11)
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.legend(title="Day", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()